# Raman Fiber Amplifier Simulator

Interactive tool for a Raman fiber amplifier that also accounts for stimulated
Brillouin scattering (SBS) suppression. Solves the coupled power-propagation
equations as a boundary value problem (`scipy.integrate.solve_bvp`).

**Features**

- Every physical/numerical parameter is entered manually (text boxes), no sliders.
- Switch between **forward** and **backward** pump propagation.
- Two modes:
  - **Simulate amplifier** — sweeps fiber length and plots signal / SBS power.
  - **Extract Raman gain coefficient** — given measured signal and pump power
    at each end of a fiber of known length, solves for `gr` (both a fast
    analytic undepleted-pump estimate, and a numerically exact fit that
    accounts for pump depletion).

Run the single cell below and use the widgets that appear.

In [3]:
import numpy as np
import scipy.integrate as intg
import scipy.optimize as opt
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# ======================================================================
# CORE PHYSICS
# ======================================================================

def raman_odes(x, y, gr, eps, gbeff, Aeff, alfap, alfas, lambdP, lambdR, pump_sign):
    """Coupled ODEs for Raman signal (Pr), pump (Pp) and SBS (Psbs) power.

    pump_sign = +1.0 -> backward pumping (pump launched at z = L)
    pump_sign = -1.0 -> forward pumping  (pump launched at z = 0)

    The sign flips the pump equation because both propagation directions
    are solved on the same fixed x-axis (0 -> L); a counter-propagating
    wave picks up a minus sign relative to a co-propagating one.
    """
    Pr, Pp, Psbs = y
    dPrdx = (eps * gr * Pp * Pr) / Aeff - (gbeff * Pr * Psbs) / Aeff - alfas * Pr
    dPpdx = pump_sign * (
        (lambdR / lambdP) * (eps * gr * Pr * Pp) / Aeff
        + (lambdR / lambdP) * (eps * gr * Pp * Psbs) / Aeff
        + alfap * Pp
    )
    dPsbsdx = -(eps * gr * Pp * Psbs) / Aeff - (gbeff * Pr * Psbs) / Aeff + alfas * Psbs
    return np.vstack((dPrdx, dPpdx, dPsbsdx))


def make_bc(Pr0, PpL, PsbsL, direction):
    """Boundary conditions.

    The signal (Pr) is always forward-propagating, seeded at z = 0.
    The SBS wave always counter-propagates the signal, seeded at z = L.
    Only the pump boundary location depends on the pumping direction.
    """
    def bc(ya, yb):
        if direction == 'forward':
            return np.array([ya[0] - Pr0, ya[1] - PpL, yb[2] - PsbsL])
        else:  # backward
            return np.array([ya[0] - Pr0, yb[1] - PpL, yb[2] - PsbsL])
    return bc


def simulate(L, dz, nodes, gr, eps, gbeff, Aeff, alfap, alfas, lambdP, lambdR,
             Pr0, PpL, PsbsL, direction):
    """Sweep fiber length from 0 to L, solving the BVP at every step.

    Uses continuation: each length is solved starting from the previous
    (converged) length's solution, interpolated onto the new grid, instead
    of a single flat guess reused for every length. This keeps the Newton
    solve well-behaved even when the true profile becomes sharply peaked
    at longer lengths (e.g. strong pump depletion / high seed power).

    Points where the solver fails to converge, or where energy conservation
    is grossly violated (a passive fiber cannot output more optical power
    than seed + pump put in), are discarded (returned as NaN) instead of
    being plotted as spurious numbers.

    Returns arrays: length, signal power at z = L, pump power at its
    launch end, SBS power at z = 0.
    """
    pump_sign = -1.0 if direction == 'forward' else 1.0
    bc = make_bc(Pr0, PpL, PsbsL, direction)
    ode = lambda x, y: raman_odes(x, y, gr, eps, gbeff, Aeff, alfap, alfas,
                                   lambdP, lambdR, pump_sign)

    lengths = np.linspace(2, L, dz)
    pr_out, pp_out, psbs_out = [], [], []
    n_failed = 0

    # Photon-number bound: total power can never exceed what was launched,
    # converted to photon flux at the two wavelengths and back. Anything
    # far beyond this is a numerical artifact, not a physical solution.
    max_plausible_power = 100 * (Pr0 + PpL * lambdR / lambdP + PsbsL)

    # initial (flat) guess for the very first, shortest length
    x_prev = np.linspace(0, lengths[0], 200)
    y_prev = np.vstack((
        np.full_like(x_prev, Pr0 if Pr0 > 0 else 1e-4),
        np.full_like(x_prev, PpL if PpL > 0 else 1e-3),
        np.full_like(x_prev, PsbsL if PsbsL > 0 else 1e-6),
    ))

    for Lcur in lengths:
        x = np.linspace(0, Lcur, dz)
        # warm-start from the previous converged solution, resampled onto
        # the new (slightly longer) grid
        y_guess = np.vstack([np.interp(x, x_prev, y_prev[i]) for i in range(3)])

        sol = intg.solve_bvp(ode, bc, x, y_guess, tol=1e-10, bc_tol=1e-10,
                              max_nodes=nodes)

        ok = sol.success and np.all(np.isfinite(sol.y)) and \
            np.max(np.abs(sol.y)) < max_plausible_power

        if ok:
            x_prev, y_prev = sol.x, sol.y  # carry forward for continuation
            pr_out.append(sol.y[0][-1])
            # Pump "output" power is read at whichever end is NOT fixed by
            # the boundary condition, i.e. the end the pump has actually
            # propagated through and been depleted along:
            #   forward pumping  -> pump launched at z=0, output at z=L
            #   backward pumping -> pump launched at z=L, output at z=0
            pp_out.append(sol.y[1][-1] if direction == 'forward' else sol.y[1][0])
            psbs_out.append(sol.y[2][0])
        else:
            n_failed += 1
            pr_out.append(np.nan)
            pp_out.append(np.nan)
            psbs_out.append(np.nan)
            # keep the last good profile as the guess for the next length
            # rather than propagating the bad one forward

    if n_failed:
        print(f"Note: the solver did not converge to a physical solution at "
              f"{n_failed} of {dz} length step(s); those points are left as "
              f"gaps in the plot. This usually means the chosen gr/Pp/Pr0 "
              f"combination produces gain too strong to resolve reliably at "
              f"that length - try more length steps, more BVP nodes, or "
              f"a lower seed/pump power.")

    return lengths, np.array(pr_out), np.array(pp_out), np.array(psbs_out)


# ======================================================================
# RAMAN GAIN COEFFICIENT EXTRACTION
# (mode: derive gr from measured signal/pump power at each fiber end)
# ======================================================================

def raman_odes_2var(x, y, gr, eps, Aeff, alfap, alfas, lambdP, lambdR, pump_sign):
    """Reduced two-wave equations (signal + pump only, no SBS) used to fit gr
    against measured boundary powers."""
    Pr, Pp = y
    dPrdx = (eps * gr * Pp * Pr) / Aeff - alfas * Pr
    dPpdx = pump_sign * ((lambdR / lambdP) * (eps * gr * Pr * Pp) / Aeff + alfap * Pp)
    return np.vstack((dPrdx, dPpdx))


def extract_gr_analytic(L, Ps_in, Ps_out, Pp_launch, eps, Aeff, alfap, alfas):
    """Fast undepleted-pump / effective-length estimate of gr.

    Standard approximation used to interpret on/off Raman gain
    measurements when the pump is much stronger than the signal.
    """
    Leff = (1 - np.exp(-alfap * L)) / alfap
    return Aeff * (np.log(Ps_out / Ps_in) + alfas * L) / (eps * Pp_launch * Leff)


def extract_gr_numeric(L, Ps_in, Ps_out, Pp_launch, eps, Aeff, alfap, alfas,
                        lambdP, lambdR, direction, gr_bounds=(1e-17, 1e-11),
                        n_scan=60):
    """Root-find gr so that the full (pump-depletion-aware) two-wave BVP
    reproduces the measured input/output signal power for the given
    pump launch power and direction.

    Signal power grows monotonically with gr, but for large enough gr the
    BVP solver can become numerically stiff and fail to converge. To stay
    robust, this first scans gr on a log grid (stopping as soon as the
    solver stops converging) and then brackets/refines the root with
    brentq inside the region where the solver is well-behaved.

    Returns None if no sign change (root) is found - widen gr_bounds, or
    check that the measured powers/length are physically consistent.
    """
    pump_sign = -1.0 if direction == 'forward' else 1.0

    def bc(ya, yb):
        if direction == 'forward':
            return np.array([ya[0] - Ps_in, ya[1] - Pp_launch])
        else:
            return np.array([ya[0] - Ps_in, yb[1] - Pp_launch])

    x = np.linspace(0, L, 200)
    y_guess = np.vstack((np.full_like(x, Ps_in), np.full_like(x, Pp_launch)))

    def output_error(gr):
        sol = intg.solve_bvp(
            lambda x, y: raman_odes_2var(x, y, gr, eps, Aeff, alfap, alfas,
                                          lambdP, lambdR, pump_sign),
            bc, x, y_guess, tol=1e-10, bc_tol=1e-10, max_nodes=20000)
        if not sol.success:
            return np.nan
        return sol.y[0][-1] - Ps_out

    lo, hi = gr_bounds
    grid = np.logspace(np.log10(lo), np.log10(hi), n_scan)

    prev_gr, prev_err = None, None
    for gr in grid:
        err = output_error(gr)
        if np.isnan(err):
            break  # solver diverged for this and all larger gr - stop scanning
        if err == 0:
            return gr
        if prev_err is not None and prev_err * err < 0:
            return opt.brentq(output_error, prev_gr, gr, xtol=1e-22, rtol=1e-12)
        prev_gr, prev_err = gr, err

    return None


# ======================================================================
# UI - every parameter is entered manually (no sliders), plus dropdowns
# for pump direction and operating mode
# ======================================================================

style = {'description_width': '170px'}
w_layout = widgets.Layout(width='340px')

mode_w = widgets.Dropdown(
    options=['Simulate amplifier', 'Extract Raman gain coefficient'],
    value='Simulate amplifier', description='Mode:', style=style, layout=w_layout)

direction_w = widgets.Dropdown(
    options=['Backward pumping', 'Forward pumping'],
    value='Backward pumping', description='Pump direction:', style=style, layout=w_layout)

# --- shared fiber / physical parameters ---
L_w        = widgets.FloatText(value=7000,    description='Fiber length L [m]',    style=style, layout=w_layout)
Aeff_w     = widgets.FloatText(value=5.2e-11, description='Aeff [m^2]',             style=style, layout=w_layout)
alfap_dB_w = widgets.FloatText(value=2e-4,    description='Pump loss [dB/m]',       style=style, layout=w_layout)
alfas_dB_w = widgets.FloatText(value=3e-4,    description='Signal loss [dB/m]',     style=style, layout=w_layout)
lambdP_w   = widgets.FloatText(value=1530,    description='Pump wavelength [nm]',   style=style, layout=w_layout)
lambdR_w   = widgets.FloatText(value=1651,    description='Signal wavelength [nm]', style=style, layout=w_layout)
eps_w      = widgets.BoundedFloatText(value=0.55, min=0, max=1, step=0.01,
                                       description='Polarization factor', style=style, layout=w_layout)

# --- "Simulate amplifier" mode parameters ---
gr_w     = widgets.FloatText(value=4.4e-14, description='Raman gain gr [m/W]',    style=style, layout=w_layout)
gbeff_w  = widgets.FloatText(value=4e-13,   description='Brillouin gain [m/W]',   style=style, layout=w_layout)
Pr0_w    = widgets.FloatText(value=6.5e-3,  description='Signal seed Pr0 [W]',    style=style, layout=w_layout)
PpL_w    = widgets.FloatText(value=4,       description='Pump launch power [W]', style=style, layout=w_layout)
PsbsL_w  = widgets.FloatText(value=1e-6,    description='SBS seed power [W]',    style=style, layout=w_layout)
dz_w     = widgets.IntText(value=100,       description='Length steps',          style=style, layout=w_layout)
nodes_w  = widgets.IntText(value=10000,     description='Max BVP nodes',         style=style, layout=w_layout)
show_pump_w = widgets.Checkbox(value=False, description='Show pump power on plot',
                                style=style, layout=w_layout)

simulate_box = widgets.VBox([gr_w, gbeff_w, Pr0_w, PpL_w, PsbsL_w, dz_w, nodes_w, show_pump_w])

# --- "Extract Raman gain coefficient" mode parameters ---
Ps_in_w     = widgets.FloatText(value=6.5e-3, description='Signal power in [W]',  style=style, layout=w_layout)
Ps_out_w    = widgets.FloatText(value=0.1,    description='Signal power out [W]', style=style, layout=w_layout)
Pp_launch_w = widgets.FloatText(value=4,      description='Pump launch power [W]',style=style, layout=w_layout)

extract_box = widgets.VBox([Ps_in_w, Ps_out_w, Pp_launch_w])

shared_box = widgets.VBox([L_w, Aeff_w, alfap_dB_w, alfas_dB_w, lambdP_w, lambdR_w, eps_w])
mode_container = widgets.VBox([simulate_box])

run_btn = widgets.Button(description='Run', button_style='primary')
out = widgets.Output()


def on_mode_change(change):
    if change['new'] == 'Simulate amplifier':
        mode_container.children = [simulate_box]
    else:
        mode_container.children = [extract_box]


mode_w.observe(on_mode_change, names='value')


def on_run_clicked(_):
    with out:
        clear_output(wait=True)
        direction = 'forward' if direction_w.value == 'Forward pumping' else 'backward'
        alfap = alfap_dB_w.value * np.log(10) / 10
        alfas = alfas_dB_w.value * np.log(10) / 10
        L = L_w.value
        Aeff = Aeff_w.value
        eps = eps_w.value
        lambdP = lambdP_w.value
        lambdR = lambdR_w.value

        if mode_w.value == 'Simulate amplifier':
            lengths, pr, pp, psbs = simulate(
                L, dz_w.value, nodes_w.value, gr_w.value, eps, gbeff_w.value, Aeff,
                alfap, alfas, lambdP, lambdR, Pr0_w.value, PpL_w.value, PsbsL_w.value,
                direction)
            plt.figure(figsize=(7, 5))
            plt.plot(lengths, pr, label='Signal power at z = L [W]')
            plt.plot(lengths, psbs, label='SBS power at z = 0 [W]')
            if show_pump_w.value:
                pump_end = 'z = L' if direction == 'forward' else 'z = 0'
                plt.plot(lengths, pp, label=f'Pump power at {pump_end} [W]')
            plt.legend()
            plt.xlabel('Amplifier length [m]')
            plt.ylabel('Optical power [W]')
            plt.xlim(0, L)
            plt.title(f'{direction.capitalize()} pumping')
            plt.show()
        else:
            gr_an = extract_gr_analytic(L, Ps_in_w.value, Ps_out_w.value, Pp_launch_w.value,
                                         eps, Aeff, alfap, alfas)
            gr_num = extract_gr_numeric(L, Ps_in_w.value, Ps_out_w.value, Pp_launch_w.value,
                                         eps, Aeff, alfap, alfas, lambdP, lambdR, direction)
            print(f"Analytic (undepleted-pump) estimate : gr = {gr_an:.4e} m/W")
            if gr_num is not None:
                print(f"Numeric (full BVP fit) result       : gr = {gr_num:.4e} m/W")
            else:
                print("Numeric fit did not converge in the default bracket "
                      "(1e-16 - 1e-11 m/W). Check the input powers/length, "
                      "or widen gr_bounds in extract_gr_numeric().")


run_btn.on_click(on_run_clicked)

ui = widgets.VBox([mode_w, direction_w, shared_box, mode_container, run_btn, out])
display(ui)
